In [1]:
import os
import glob
import pandas as pd
import numpy as np

# 设置输入和输出文件夹路径
input_dir = r"F:\self_quant\data\data\合并后数据"
output_dir = r"F:\self_quant\data\data\合并后数据_带市值"

# 如果输出文件夹不存在，则创建它以存放处理后的数据
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 获取输入文件夹下所有的 csv 文件
file_pattern = os.path.join(input_dir, "*.csv")
file_list = glob.glob(file_pattern)

print(f"共找到 {len(file_list)} 个CSV文件，开始批量处理...")

# 计数器
success_count = 0

for file_path in file_list:
    file_name = os.path.basename(file_path)
    output_path = os.path.join(output_dir, file_name)

    try:
        # 读取 CSV 数据
        # (国内炒股软件导出的CSV大多是GBK编码，如果运行报错说编码问题，可以把 'gbk' 改成 'utf-8')
        df = pd.read_csv(file_path, encoding='gbk')

        # 检查是否包含计算所需的列
        if 'volume' in df.columns and 'turn' in df.columns and 'close' in df.columns:

            # 1. 避免除以0报错，将换手率为0的替换为空值(NaN)
            df['turn'] = df['turn'].replace(0, np.nan)

            # 2. 计算流通股本
            # 根据你截图的 turn 为 0.83... ，推测它是去掉了 % 号的百分比，因此换算成真实比例需要除以 100
            df['float_shares'] = df['volume'] / (df['turn'] / 100)

            # 3. 计算流通市值 = 流通股本 * 收盘价
            df['float_mv'] = df['float_shares'] * df['close']

        else:
            print(f"⚠️ 文件 {file_name} 缺少必要的列 (volume, turn, close)，无法计算流通市值。")

        # 将带有 'float_mv' 列的新数据保存到输出文件夹
        df.to_csv(output_path, index=False, encoding='gbk')
        success_count += 1

        # 每处理 500 个文件打印一次进度
        if success_count % 500 == 0:
            print(f"已处理 {success_count} 个文件...")

    except Exception as e:
        print(f"❌ 处理文件 {file_name} 时出错: {e}")

print(f"\n全部处理完成！成功处理了 {success_count} 个文件。")
print(f"带有流通市值(float_mv)的新数据已保存至：{output_dir}")

共找到 5443 个CSV文件，开始批量处理...
已处理 500 个文件...
已处理 1000 个文件...
已处理 1500 个文件...
已处理 2000 个文件...
已处理 2500 个文件...
已处理 3000 个文件...
已处理 3500 个文件...
已处理 4000 个文件...
已处理 4500 个文件...
已处理 5000 个文件...

全部处理完成！成功处理了 5443 个文件。
带有流通市值(float_mv)的新数据已保存至：F:\self_quant\data\data\合并后数据_带市值
